<a href="https://colab.research.google.com/github/sjlee53/ESAA-submission/blob/main/project/0920_ESAA_YB_movie_visualization_candidates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ESAA YB 미니프로젝트 — 시각화 후보

DACON 영화 관객수 데이터(2010~2015년 한국영화 600편)로 세 가지 가설을 단계적으로 검증한다.

| | 주제 | 결론 |
|---|---|---|
| 3번 | 작은 영화일수록 상영시간이 딱 떨어진다 | 경향은 확인되나 통계적으로는 경계선 |
| 6번 | 인력과 관객 사이에는 문턱이 있다 | 스태프 117~279명 구간에 증가가 몰림 |
| 8번 | 제작 규모에 따라 장르의 영향력이 달라진다 | 대규모에서 장르 차이가 가장 작음 |

세 분석 모두 **묶어서 보면 안 보이고, 쪼개서 보면 결론이 바뀐다**는 구조를 가진다.

## 0. 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

url = 'https://raw.githubusercontent.com/sjlee53/ESAA-submission/main/project/data/movies_train.csv'
df = pd.read_csv(url)
print(df.shape)
df.head()

In [ ]:
# 코랩 한글 폰트 (한 번만 실행)
!apt-get -qq install fonts-nanum
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')

sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'NanumGothic'   # seaborn이 폰트를 덮으므로 set_style 뒤에
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 전처리
df['dir_prev_bfnum'] = df['dir_prev_bfnum'].fillna(0)
df['release_time'] = pd.to_datetime(df['release_time'])
df['연도'] = df['release_time'].dt.year
df['월'] = df['release_time'].dt.month

df['규모'] = pd.cut(df['num_staff'], [-1, 80, 250, 900],
                  labels=['소(~80명)', '중(81~250명)', '대(251명~)'])
df['감독'] = np.where(df['dir_prev_num'] == 0, '신인', '경력')
df['log관객수'] = np.log10(df['box_off_num'])

df[['title', 'genre', 'time', 'num_staff', '규모', 'box_off_num']].head()

---

# 3번. 작은 영화일수록 상영시간이 딱 떨어진다

**가설**: 제작 규모가 작은 영화일수록 상영시간이 90분·95분처럼 5의 배수에 몰릴 것이다.

상영시간이 5로 끝날 이유는 없다. 그런데도 쏠림이 있다면 실제 길이가 아니라 **기재 방식**이
남긴 흔적일 수 있다.

### 1단계. 전체에 쏠림이 있는가 — 기준선과 비교

상영시간이 무작위라면 5의 배수는 20%, 10의 배수는 10%여야 한다. 이 기준선과 비교한다.

In [ ]:
print('5의 배수 : %.1f%%  (무작위면 20%%)' % ((df['time'] % 5 == 0).mean() * 100))
print('10의 배수: %.1f%%  (무작위면 10%%)' % ((df['time'] % 10 == 0).mean() * 100))
print()
print('가장 흔한 상영시간 10개')
print(df['time'].value_counts().head(10).to_string())

### 2단계. 제작 규모에 따라 갈리는가

소 → 대로 갈수록 비율이 줄어든다면 가설과 맞는다.

In [ ]:
df.groupby('규모', observed=True).agg(
    편수=('time', 'size'),
    배수5=('time', lambda s: round((s % 5 == 0).mean() * 100, 1)),
    배수10=('time', lambda s: round((s % 10 == 0).mean() * 100, 1)),
    상영시간중앙=('time', 'median'))

### 3단계. 0 때문인가 5 때문인가 — 끝자리를 펼쳐서 확인

"5의 배수가 많다"만으로는 어느 자리가 솟았는지 알 수 없다.

In [ ]:
df['끝자리'] = df['time'] % 10
(pd.crosstab(df['규모'], df['끝자리'], normalize='index') * 100).round(1)

### 4단계. 우연인가 — 카이제곱 검정

In [ ]:
tab = pd.crosstab(df['규모'], df['time'] % 5 == 0)
print(tab.to_string())
print()
print('소·중·대 전체 p = %.4f' % stats.chi2_contingency(tab)[1])
print('소 vs 대    p = %.4f' % stats.chi2_contingency(tab.iloc[[0, 2]])[1])

### 5단계. 차이의 크기 — 관측 편수 vs 기대 편수

비율(%)만으로는 체감이 어려우므로 실제 편수로 환산한다.

In [ ]:
for 규모 in df['규모'].cat.categories:
    d = df[df['규모'] == 규모]
    관측 = (d['time'] % 5 == 0).sum()
    기대 = len(d) * 0.2
    print('%-12s %3d편 중 5의 배수 %2d편  (기대 %4.1f편, 차이 %+.1f)'
          % (규모, len(d), 관측, 기대, 관측 - 기대))

### 6단계. 장르 구성 때문은 아닌가 — 교란변수 통제

소규모에는 다큐멘터리가 90편 있지만 중규모에는 3편, 대규모에는 0편이다.
그런데 **다큐멘터리는 원래 5의 배수 비율이 높다.** 그렇다면 소규모의 높은 비율이
"소규모라서"가 아니라 **"다큐가 많아서"** 생긴 것일 수 있다.

In [ ]:
# 장르마다 5의 배수 비율이 다른지 먼저 확인
t = df.groupby('genre').agg(편수=('time', 'size'),
                            배수5=('time', lambda s: round((s % 5 == 0).mean() * 100, 1)))
print(t.sort_values('배수5', ascending=False).to_string())
print()
print('[소규모의 장르 구성]')
print(df[df['규모'] == '소(~80명)']['genre'].value_counts().head(5).to_string())

In [ ]:
# 장르를 통제하고 다시 계산
def 배수5비율(데이터, 라벨):
    g = 데이터.groupby('규모', observed=True)['time'].agg(
        편수='size', 비율=lambda s: round((s % 5 == 0).mean() * 100, 1))
    print('[%s]' % 라벨)
    print(g.to_string())
    print()


배수5비율(df, '전체')
배수5비율(df[df['genre'] != '다큐멘터리'], '다큐멘터리 제외')
배수5비율(df[df['genre'] == '드라마'], '드라마만')

### 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6),
                         gridspec_kw={'width_ratios': [1.55, 1]})

# ── 왼쪽: 소규모 끝자리 분포 (0·5 강조) ──
소 = df[df['규모'] == '소(~80명)']
비율 = 소['끝자리'].value_counts(normalize=True).reindex(range(10), fill_value=0) * 100
색 = ['#d62728' if i in (0, 5) else '#c9ccd1' for i in range(10)]

axes[0].bar(비율.index, 비율.values, color=색, width=0.68)
axes[0].axhline(10, color='black', linestyle='--', linewidth=1.3,
                label='무작위 기대치 10%')
for i, v in zip(비율.index, 비율.values):
    axes[0].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=9,
                 color='#d62728' if i in (0, 5) else '#999')
axes[0].set_xticks(range(10))
axes[0].set_xlabel('상영시간의 끝자리', fontsize=12)
axes[0].set_ylabel('비율 (%)', fontsize=12)
axes[0].set_title(f'소규모 영화 {len(소)}편: 5의 배수 끝자리(0, 5)가 기대보다 많다',
                  fontsize=13)
axes[0].set_ylim(0, 18)
axes[0].legend(loc='upper right', fontsize=10)

# ── 오른쪽: 관측 - 기대 편수 (0선 교차) ──
diff = df.groupby('규모', observed=True)['time'].apply(
    lambda s: (s % 5 == 0).sum() - len(s) * 0.2)
색2 = ['#d62728' if v > 0 else '#1f77b4' for v in diff.values]

axes[1].bar(range(3), diff.values, color=색2, width=0.58)
axes[1].axhline(0, color='black', linewidth=1.4)
for i, v in enumerate(diff.values):
    axes[1].text(i, v + (1.2 if v > 0 else -2.4), f'{v:+.1f}편',
                 ha='center', fontsize=12, fontweight='bold',
                 color='#d62728' if v > 0 else '#1f77b4')
axes[1].set_xticks(range(3))
axes[1].set_xticklabels(diff.index, fontsize=9)
axes[1].set_ylabel('기대 대비 편수 차이', fontsize=12)
axes[1].set_title('기대보다 몇 편이나 많은가', fontsize=13)
axes[1].set_ylim(-8, 30)

fig.suptitle('작은 영화일수록 상영시간이 딱 떨어진다', fontsize=16, y=0.99)
plt.tight_layout()
plt.show()

**해석**

소규모 영화는 상영시간 끝자리 0과 5가 각각 14.1%로 가장 높다. 나머지 여덟 자리는
6.7~11.1%에 흩어져 있다. 두 자리가 동시에 솟았다는 것은 "5 단위로 끊어 적는다"는
하나의 원인을 가리킨다.

기대치 대비 편수로 보면 소 **+24.6편**, 중 +5.4편, 대 **−2.0편**이다. 규모가 커질수록
줄어들고, 대규모는 오히려 기준치보다 적다.

**장르 구성 때문은 아니다.** 소규모에는 5의 배수 비율이 높은 다큐멘터리(32.3%)가 90편
몰려 있지만, 다큐를 빼도 소 26.6% / 중 23.7% / 대 18.8%로 순서가 유지되고,
드라마만 봐도 소 27.7% / 중 24.2% / 대 14.8%로 같다.

상영시간이 5로 끝날 이유는 없다. 실제 길이가 그런 것이 아니라 **작은 영화일수록
상영시간을 반올림해서 기재했을 가능성**이 크다.

**한계**: 소 vs 대는 p=0.032로 유의하지만 세 그룹 전체로는 **p=0.074**로 0.05를 넘지
못한다. 확정이 아니라 **경향**으로 보아야 한다. 또 0·5가 높은 것은 사실이나
대규모의 4(15.2%)처럼 다른 자리도 흔들리므로, "0과 5만 솟는다"고까지는 말할 수 없다.
그리고 "반올림했다"는 직접 확인한 것이 아니라 추론이다.

---

# 6번. 인력과 관객 사이에는 문턱이 있다

**가설**: "큰 영화가 스태프 1명당 관객수(효율)도 24배 높다"는 말이 사실이라면,
제작 규모가 커질수록 효율이 꾸준히 올라야 한다.

효율은 `관객수 ÷ 스태프수`로 정의한다. 다만 **분모가 작으면 값이 자동으로 커지므로**
지표 자체를 의심하면서 확인한다.

### 1단계. 24배가 사실인가 — 규모별 효율

In [ ]:
d = df[df['num_staff'] > 0].copy()
d['효율'] = d['box_off_num'] / d['num_staff']
print('스태프 0명이라 제외한 영화: %d편' % (len(df) - len(d)))
print()

print(d.groupby('규모', observed=True).agg(
    편수=('효율', 'size'),
    스태프중앙=('num_staff', 'median'),
    관객중앙=('box_off_num', 'median'),
    효율중앙=('효율', 'median')).round(0).to_string())

소 = d[d['규모'] == '소(~80명)']
대 = d[d['규모'] == '대(251명~)']
print()
print('효율 %.1f배' % (대['효율'].median() / 소['효율'].median()))

### 2단계. 그 24배는 어디서 왔나 — 분모와 분자로 쪼개기

In [ ]:
스태프배수 = 대['num_staff'].median() / 소['num_staff'].median()
관객배수 = 대['box_off_num'].median() / 소['box_off_num'].median()
print('스태프 %.1f배  →  관객 %.1f배' % (스태프배수, 관객배수))

### 3단계. 단조 증가인가 U자인가

소·중·대 3칸으로는 안이 뭉개지므로 10등분해서 모양을 본다.

In [ ]:
d['분위'] = pd.qcut(d['num_staff'], 10, labels=False) + 1
분위표 = d.groupby('분위').agg(
    편수=('효율', 'size'),
    스태프중앙=('num_staff', 'median'),
    관객중앙=('box_off_num', 'median'),
    효율중앙=('효율', 'median')).round(0)
print(분위표.to_string())

바닥 = 분위표['효율중앙'].idxmin()
print()
print('효율 최저: %d분위 (스태프 %.0f명, 1명당 %.0f명)'
      % (바닥, 분위표.loc[바닥, '스태프중앙'], 분위표.loc[바닥, '효율중앙']))
print('단조 증가인가:', 분위표['효율중앙'].is_monotonic_increasing)

### 4단계. 분모 함정인가 — 효율 상위가 어떤 영화들인가

In [ ]:
print('[효율 상위 5편]')
print(d.nlargest(5, '효율')[['title', 'genre', 'num_staff',
                            'box_off_num', '효율']].round(0).to_string(index=False))
print()
print('[대규모인데 효율 하위 3편]')
print(대.nsmallest(3, '효율')[['title', 'genre', 'num_staff',
                             'box_off_num', '효율']].round(0).to_string(index=False))

### 시각화

In [ ]:
t = d.groupby('분위').agg(스태프중앙=('num_staff', 'median'),
                        관객중앙=('box_off_num', 'median'))
t['배수'] = t['관객중앙'] / t['관객중앙'].shift(1)   # 앞 분위 대비
라벨 = [f'{i}\n({s:.0f}명)' for i, s in zip(t.index, t['스태프중앙'])]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.8), sharex=True)

# ── 왼쪽: 관객수 곡선 ──
ax = axes[0]
ax.axvspan(0.5, 4.5, color='#ffd9d9', alpha=0.5, zorder=0)   # 평평한 구간
ax.plot(t.index, t['관객중앙'], color='#4c72b0', linewidth=2.2, zorder=2)
ax.scatter(t.index, t['관객중앙'], s=95, color='#4c72b0', zorder=3)
ax.set_yscale('log')
ax.text(2.5, 250, '스태프 3 → 44명 (15배)\n관객은 그대로',
        fontsize=11, fontweight='bold', color='#c0392b', ha='center')
ax.set_ylim(150, 6e6)
ax.set_ylabel('관객 수 중앙값 (로그)', fontsize=12)
ax.set_title('① 스태프 44명까지는 관객이 늘지 않는다', fontsize=13)

# ── 오른쪽: 앞 분위 대비 증가 배수 (1분위는 앞 구간이 없어 계산 불가) ──
ax = axes[1]
유효 = t.dropna(subset=['배수'])
색 = ['#d62728' if v >= 5 else '#b0bec5' for v in 유효['배수']]
ax.bar(유효.index, 유효['배수'], color=색, width=0.66)
ax.axhline(1, color='black', linestyle='--', linewidth=1.3, label='1.0배 = 변화 없음')
ax.text(1, 0.45, '앞 구간\n없음', ha='center', va='center', fontsize=9, color='#999')
for i, v in zip(유효.index, 유효['배수']):
    ax.text(i, v + 0.35, f'{v:.1f}배', ha='center', fontsize=10,
            fontweight='bold' if v >= 5 else 'normal',
            color='#d62728' if v >= 5 else '#777')
ax.set_ylim(0, 14.5)
ax.set_ylabel('앞 분위 대비 관객 증가 배수', fontsize=12)
ax.set_title('② 문턱은 스태프 117 → 279명 구간', fontsize=13)
ax.legend(loc='upper left', fontsize=10)

for ax in axes:
    ax.set_xticks(t.index)
    ax.set_xticklabels(라벨, fontsize=9)
    ax.set_xlabel('스태프 수 10분위 (괄호는 구간 중앙값)', fontsize=11)

fig.suptitle('인력과 관객 사이에는 문턱이 있다', fontsize=16, y=1.0)
plt.tight_layout()
plt.show()

**해석**

스태프가 3명에서 44명으로 **15배 늘어도 관객은 그대로**다(2,148 → 1,718명).
로그 축인데도 선이 수평이다.

증가 배수로 보면 2~4분위는 0.7 / 1.2 / 0.9배로 1 근처에 머물다가, 7분위 6.7배,
8분위 **12.0배**로 두 칸만 솟고 다시 1.6~1.7배로 내려앉는다. 증가가
**스태프 117~279명 한 구간에 몰려 있다.**

인력을 조금씩 늘려 관객이 조금씩 느는 구조가 아니라 **문턱이 있는 구조**다.
효율(스태프 1명당 관객)이 단조 증가가 아니라 U자인 것도 여기서 나온다 — 문턱 아래에서는
분모만 커지고 분자는 그대로라 효율이 떨어지고, 문턱을 넘으면 분자가 크게 늘어 효율이 올라간다.

**한계 1 — 지표의 함정**: "큰 영화가 24배 효율적"은 맞지만 오해를 낳는다. 효율 상위 5편 중
3편이 스태프 1~3명(님아 3명, 울지마 톤즈 1명, 뽀로로 2명)이다. 분모가 작으면 값이 자동으로 커진다.

**한계 2 — 인과가 아니다**: 인력을 늘려서 관객이 는 것이 아니라, **흥행이 예상되는 작품에
인력이 투입된 것**일 수 있다. 또 제작비 변수가 없어 스태프 수를 제작 규모의 대리 지표로 사용했다.

---

# 8번. 제작 규모에 따라 장르의 영향력이 달라진다

**가설**: 제작 규모를 고정하면, 장르가 관객수를 가르는 정도가 규모마다 다를 것이다.

주의할 점이 있다. **규모마다 장르 구성 자체가 다르다.** 대규모에는 다큐멘터리와
애니메이션이 한 편도 없다. 구성을 맞추지 않으면 "장르 효과"가 아니라 "구성 차이"를 재게 된다.

### 0단계. 지표 계산 함수

같은 계산을 데이터만 바꿔 두 번 돌리므로 함수로 묶는다. 잣대가 같아야 비교가 공정하다.

- **최고최저배수**: 장르별 관객수 중앙값의 최대 ÷ 최소
- **R2**: 관객수(로그) 분산 중 장르가 설명하는 비율
- **간편차**: 장르별 로그 중앙값들의 표준편차 (장르끼리 얼마나 다른가)
- **내편차**: 같은 장르 안 로그 관객수 표준편차의 평균 (장르 안에서 얼마나 흩어지는가)

In [ ]:
주요장르 = df['genre'].value_counts()
주요장르 = 주요장르[주요장르 >= 20].index.tolist()
sub = df[df['genre'].isin(주요장르)].copy()


def 장르효과(데이터, 최소편수=5):
    """규모별로 '장르가 관객수를 얼마나 가르는가'를 네 지표로 계산."""
    결과 = []
    for 규모 in 데이터['규모'].cat.categories:
        g = 데이터[데이터['규모'] == 규모]
        편수 = g['genre'].value_counts()
        g = g[g['genre'].isin(편수[편수 >= 최소편수].index)]   # 표본 적은 장르 제외
        if g['genre'].nunique() < 2:
            continue

        묶음 = g.groupby('genre', observed=True)
        중앙 = 묶음['box_off_num'].median()
        그룹 = [v['log관객수'].values for _, v in 묶음]

        전체 = np.concatenate(그룹)
        ssb = sum(len(a) * (a.mean() - 전체.mean()) ** 2 for a in 그룹)   # 장르 간 변동
        sst = ((전체 - 전체.mean()) ** 2).sum()                          # 전체 변동

        결과.append({
            '규모': 규모,
            '편수': len(전체),
            '장르수': len(그룹),
            '최고최저배수': round(중앙.max() / 중앙.min(), 1),
            'R2': round(ssb / sst, 3),
            'p': round(stats.f_oneway(*그룹)[1], 4),
            '간편차': round(묶음['log관객수'].median().std(), 3),
            '내편차': round(묶음['log관객수'].std().mean(), 3),
        })
    return pd.DataFrame(결과).set_index('규모')

### 1단계. 규모마다 장르 구성이 같은가 — 먼저 확인할 것

In [ ]:
구성 = pd.crosstab(sub['규모'], sub['genre'])
구성

### 2단계. [1차] 주요 장르 전체로 계산 — 교란이 섞인 결과

In [ ]:
일차 = 장르효과(sub)
일차

### 3단계. [교정] 세 규모 모두에 5편 이상 있는 공통 장르만

1단계에서 대규모에 다큐·애니가 0편인 것을 확인했으므로 장르 구성을 맞춘다.

In [ ]:
공통장르 = 구성.columns[(구성 >= 5).all(axis=0)].tolist()
print('공통 장르:', 공통장르)
print()

공통 = sub[sub['genre'].isin(공통장르)].copy()
이차 = 장르효과(공통)
print(이차.to_string())

### 4단계. 1차 vs 교정 — 결론이 바뀌는가

In [ ]:
비교 = pd.DataFrame({
    '1차_배수': 일차['최고최저배수'], '교정_배수': 이차['최고최저배수'],
    '1차_R2': 일차['R2'], '교정_R2': 이차['R2'],
    '1차_p': 일차['p'], '교정_p': 이차['p']})
비교

### 5단계. 시각화에 쓸 표 — 공통 장르의 규모별 중앙값

In [ ]:
표 = 공통.pivot_table(index='규모', columns='genre',
                     values='box_off_num', aggfunc='median', observed=True)
print(표.round(0).to_string())
print()
print('[중규모의 장르별 편수]')
print(공통[공통['규모'] == '중(81~250명)']['genre'].value_counts().to_string())

### 시각화 ① — 교정으로 무엇이 바뀌었나

In [ ]:
x = np.arange(3)
w = 0.36

fig, ax = plt.subplots(figsize=(10, 5.4))
ax.bar(x - w / 2, 일차['최고최저배수'], w, label='1차 (주요 장르 전체)', color='#b0bec5')
ax.bar(x + w / 2, 이차['최고최저배수'], w, label='교정 (공통 장르 4개)', color='#d62728')

for i in x:
    ax.text(i - w / 2, 일차['최고최저배수'].iloc[i] + 1.3,
            f"{일차['최고최저배수'].iloc[i]:.1f}배", ha='center', fontsize=11, color='#666')
    ax.text(i + w / 2, 이차['최고최저배수'].iloc[i] + 1.3,
            f"{이차['최고최저배수'].iloc[i]:.1f}배", ha='center', fontsize=11,
            fontweight='bold', color='#d62728')

ax.set_xticks(x)
ax.set_xticklabels(일차.index, fontsize=11)
ax.set_ylim(0, 58)
ax.set_ylabel('장르별 관객수 중앙값의 최고/최저 배수', fontsize=11)
ax.set_title('장르 구성을 맞추면 소규모의 45배가 3.1배로 내려앉는다', fontsize=14, pad=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 시각화 ② — 규모별 장르 격차

In [ ]:
간편차 = 이차['간편차']
배수 = 이차['최고최저배수']
x = np.arange(3)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.8),
                         gridspec_kw={'width_ratios': [1.35, 1]})

# ── 왼쪽: 장르별 선 ──
ax = axes[0]
ax.axvspan(0.6, 1.4, color='#fff3cd', alpha=0.8, zorder=0)
색 = {'공포': '#d62728', '드라마': '#4c72b0',
     '멜로/로맨스': '#e377c2', '코미디': '#2ca02c'}
for g in 공통장르:
    ax.plot(x, 표[g].values, marker='o', markersize=8,
            linewidth=2.2, label=g, color=색[g], zorder=2)
ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(표.index, fontsize=10)
ax.set_xlim(-0.35, 2.35)
ax.set_ylim(300, 3e7)
ax.set_ylabel('관객 수 중앙값 (로그)', fontsize=12)
ax.set_title('① 대규모에서는 장르 차이가 작아진다', fontsize=13)
ax.legend(fontsize=9, loc='lower right')
for i in x:
    ax.text(i, 1.1e7, f'{배수.iloc[i]:.1f}배', ha='center', fontsize=13,
            fontweight='bold', color='#c0392b' if i == 1 else '#666')

# ── 오른쪽: 장르 간 편차 ──
ax = axes[1]
ax.bar(x, 간편차.values, color=['#b0bec5', '#d62728', '#b0bec5'], width=0.58)
for i, v in enumerate(간편차.values):
    ax.text(i, v + 0.025, f'{v:.3f}', ha='center', fontsize=12,
            fontweight='bold', color='#d62728' if i == 1 else '#666')
for i, p in enumerate(이차['p']):
    꼬리 = '\n(유의하지 않음)' if p >= 0.05 else ''
    ax.text(i, 0.84, f'p={p:.4f}{꼬리}', ha='center', fontsize=9, color='#666')
ax.set_xticks(x)
ax.set_xticklabels(간편차.index, fontsize=10)
ax.set_ylim(0, 0.95)
ax.set_ylabel('장르 간 편차 (로그 관객수의 표준편차)', fontsize=12)
ax.set_title('② 장르 간 차이는 대규모에서 가장 작다', fontsize=13)

fig.suptitle('공통 장르 4개로 보면 — 대규모에서는 장르 차이가 작아진다',
             fontsize=16, y=1.0)
plt.tight_layout()
plt.show()

**해석**

세 규모 모두에 5편 이상 있는 **공통 장르 4개**(공포·드라마·멜로/로맨스·코미디)로 맞추면,
장르에 따른 관객수 격차는 소규모 **3.1배**, 중규모 **47.3배**, 대규모 **2.8배**다.

제한 없이 봤을 때 소규모가 45배로 나온 것은 **소규모에만 있는 다큐멘터리(90편)·
애니메이션(16편)** 때문이었다. 대규모에는 두 장르가 0편이라 애초에 비교가 성립하지 않는다.
교정하니 45 → 3.1배로 내려앉았고, **중규모만 47.3배 그대로**였다. 교정이 중규모를 봐준 것이
아니라 나머지 둘이 착시였다.

왼쪽 그림에서 네 장르 선은 소규모에서 붙어 있다가 중규모에서 벌어지고 대규모에서 다시 모인다.
대규모에서는 순서까지 뒤집힌다 — 중규모 1등이던 공포가 꼴찌(61만)가 되고, 꼴찌였던 드라마가
81만으로 올라선다.

**장르 차이가 유의하지 않은 것은 대규모뿐이다(p=0.2966).** 소규모(3.1배, p=0.0095)와
중규모(47.3배, p=0.0156)에서는 장르 간 차이가 남는다.

**결론: 장르 선택의 영향은 대규모에서 가장 작다.** 소·중규모에서는 장르에 따라
관객수 차이가 남는다.

**한계**: 중규모의 공포는 13편, 코미디는 12편으로 표본이 얇아 47.3배는 참고치로 보아야 한다.
또 장르는 영화당 하나만 기록되어 있어 여러 장르에 걸친 영화는 반영되지 않는다.